# CP2 · Notebook 03 — MiDaS sobre el dataset completo

## Objetivo

Aplicar **MiDaS small** a las ~21 imágenes (14 CP1 + 7 CP2 extras), medir tiempos en CPU, visualizar depth maps en grid, y guardar outputs para el notebook 04.

**Tiempo estimado**: 8 min. La inferencia es lo lento (~1–3 s/imagen × 21 ≈ 30–60 s).

**Requiere**: `02_setup.ipynb` ejecutado.

## 0. Imports y carga del modelo

In [ ]:
import time, json
from pathlib import Path
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

CP1_DATA   = Path('../../CP1-carriles/datasets/lanes-subset')
CP2_EXTRAS = Path('../datasets/cp2-depth-extras')
OUT_DIR    = Path('../outputs'); OUT_DIR.mkdir(exist_ok=True)

MIDAS_ID = 'Intel/dpt-swinv2-tiny-256'
midas_processor = AutoImageProcessor.from_pretrained(MIDAS_ID)
midas_model = AutoModelForDepthEstimation.from_pretrained(MIDAS_ID).eval()
print(f'✅ MiDaS small cargado ({sum(p.numel() for p in midas_model.parameters())/1e6:.1f} M params)')

## 1. Función de inferencia + warmup

La primera inferencia siempre es más lenta por allocaciones internas — la descartamos del timing.

In [ ]:
def midas_infer(image_pil):
    """PIL Image → (depth_map ndarray, time_ms). Depth = inverse depth, sin unidades."""
    t0 = time.perf_counter()
    inputs = midas_processor(images=image_pil, return_tensors='pt')
    with torch.no_grad():
        out = midas_model(**inputs).predicted_depth
    depth = out[0].numpy()
    return depth, (time.perf_counter() - t0) * 1000

# Warmup: 1 inferencia que descartamos
sample = next((CP1_DATA / 'easy').glob('*.png'))
_ = midas_infer(Image.open(sample).convert('RGB'))
print('warmup completado')

## 2. Recolectar las ~21 imágenes

Las catalogamos por origen + sub-categoría (easy/medium/hard del CP1 + extras de CP2).

In [ ]:
image_set = []
for cat in ['easy', 'medium', 'hard']:
    for p in sorted((CP1_DATA / cat).glob('*.png')):
        image_set.append({'path': p, 'origin': 'CP1', 'category': cat, 'name': p.name})
for p in sorted(CP2_EXTRAS.glob('*.jpg')) + sorted(CP2_EXTRAS.glob('*.png')):
    image_set.append({'path': p, 'origin': 'CP2_extras', 'category': 'challenge', 'name': p.name})

print(f'Total imágenes: {len(image_set)}')
for cat in ['easy', 'medium', 'hard', 'challenge']:
    n = sum(1 for i in image_set if i['category'] == cat)
    print(f'  {cat:10s}  {n}')

## 3. Inferencia sobre todas las imágenes

Estimado: ~1–3 s/imagen × 21 ≈ 30–60 s en CPU moderna.

In [ ]:
from tqdm import tqdm

results = []
for item in tqdm(image_set, desc='MiDaS'):
    img = Image.open(item['path']).convert('RGB')
    depth, ms = midas_infer(img)
    results.append({**item, 'depth': depth, 'time_ms': ms, 'orig_size': img.size})

times = [r['time_ms'] for r in results]
print(f'\nTiempo medio: {np.mean(times):.0f} ms · p95: {np.percentile(times, 95):.0f} ms')
print(f'Total inferencia: {sum(times)/1000:.1f} s')

## 4. Grid visual — 1 ejemplo de cada categoría

**Importante con los colormaps**: usamos `plasma` (también vale `viridis`, `magma`). **Evitar `jet`** — los humanos lo leemos mal.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
for row, cat in enumerate(['easy', 'medium', 'hard', 'challenge']):
    cat_results = [r for r in results if r['category'] == cat]
    if not cat_results:
        axes[row, 0].axis('off'); axes[row, 1].axis('off')
        continue
    r = cat_results[0]
    img = Image.open(r['path']).convert('RGB')
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"{cat} · {r['name']}")
    axes[row, 0].axis('off')
    im = axes[row, 1].imshow(r['depth'], cmap='plasma')
    axes[row, 1].set_title(f"MiDaS depth · {r['time_ms']:.0f} ms · range [{r['depth'].min():.1f}, {r['depth'].max():.1f}]")
    axes[row, 1].axis('off')
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_DIR / '03_midas_grid.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'✅ Grid guardado en {OUT_DIR / "03_midas_grid.png"}')

## 5. Distribución de tiempos

Histograma por categoría — ¿el tiempo depende del contenido de la imagen? Spoiler: **no demasiado**, porque la red procesa siempre el mismo número de píxeles tras el resize interno.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for cat, color in zip(['easy','medium','hard','challenge'], ['#4a6fa5','#f4a261','#e76f51','#7b2cbf']):
    times_cat = [r['time_ms'] for r in results if r['category'] == cat]
    if times_cat:
        ax.hist(times_cat, bins=10, alpha=0.6, label=f'{cat} ({len(times_cat)})', color=color, edgecolor='black')
ax.set_xlabel('Tiempo de inferencia (ms)')
ax.set_ylabel('Nº imágenes')
ax.set_title('MiDaS small — tiempo CPU por categoría')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / '03_midas_times.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Guardar resultados para 04/05

Guardamos los depth maps como `.npy` para que `04_relativa_vs_metrica.ipynb` y `05_depth_anything_v2.ipynb` los carguen sin re-inferir.

In [ ]:
DEPTH_DIR = OUT_DIR / '03_midas_depth'
DEPTH_DIR.mkdir(exist_ok=True)

summary = {
    'model': MIDAS_ID,
    'mean_time_ms': float(np.mean(times)),
    'median_time_ms': float(np.median(times)),
    'p95_time_ms': float(np.percentile(times, 95)),
    'total_images': len(results),
    'per_image': [],
}
for r in results:
    npy_name = f"{r['origin']}_{r['category']}_{r['name']}.npy"
    np.save(DEPTH_DIR / npy_name, r['depth'])
    summary['per_image'].append({
        'name': r['name'], 'origin': r['origin'], 'category': r['category'],
        'time_ms': float(r['time_ms']),
        'depth_min': float(r['depth'].min()),
        'depth_max': float(r['depth'].max()),
        'depth_mean': float(r['depth'].mean()),
        'npy_file': npy_name,
    })

with open(OUT_DIR / '03_midas_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'✅ {len(results)} depth maps guardados en {DEPTH_DIR}/')
print(f'✅ Resumen en {OUT_DIR / "03_midas_summary.json"}')

## 7. Reflexión rápida

Mira tu grid y responde mentalmente (lo escribirás en `07_preguntas.md`):

1. ¿En qué tipo de imagen el depth map parece "correcto" geométricamente? ¿Carriles convergen al horizonte como esperarías?
2. ¿En qué imagen el depth parece **sospechoso**? Apunta cuál.
3. El rango de valores `[min, max]` cambia entre imágenes — ¿por qué? ¿Qué implica para usar este depth en producción?

Si tienes intuición clara de la respuesta a la 3, ya entiendes la trampa del depth relativo. Pasa a `04`.